In [27]:
import numpy as np


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def l_sigmoid(w):
    w0, w1 = w
    val1 = np.clip(1.0 - sigmoid(w0 - w1), 1e-15, None)
    val2 = np.clip(sigmoid(w0 + w1), 1e-15, None)
    return -np.log(val1) - np.log(val2)


def grad_l_sigmoid(w):
    w0, w1 = w
    return np.array([
        sigmoid(w0 - w1) + sigmoid(w0 + w1) - 1.0,
        sigmoid(w0 + w1) - sigmoid(w0 - w1) - 1.0
    ])


def F1(x):
    x1, x2 = x
    return x1**2 + x2**2 - 2*x1 - 4*x2 - 1


def grad_F1(x):
    x1, x2 = x
    return np.array([2*x1 - 2, 2*x2 - 4])


def F2(x):
    x1, x2 = x
    return 3*x1**2 - 12*x1 + 2*x2**2 + 16*x2 - 10


def grad_F2(x):
    x1, x2 = x
    return np.array([6*x1 - 12, 4*x2 + 16])


def F3(x):
    x1, x2 = x
    return x1**2 - 4*x1*x2 + 4.5*x2**2 - 4*x2 + 3


def grad_F3(x):
    x1, x2 = x
    return np.array([2*x1 - 4*x2, -4*x1 + 9*x2 - 4])


def F4(x):
    x1, x2 = x
    return x1**2*x2 - 2*x1*x2**2 + 3*x1*x2 + 4


def grad_F4(x):
    x1, x2 = x
    return np.array([
        2*x1*x2 - 2*x2**2 + 3*x2,
        x1**2 - 4*x1*x2 + 3*x1
    ])


def numeric_grad(F, x, h=1e-5):
    n = len(x)
    grad = np.zeros(n)
    for i in range(n):
        xp2 = x.copy(); xp2[i] += 2*h
        xp1 = x.copy(); xp1[i] += h
        xm1 = x.copy(); xm1[i] -= h
        xm2 = x.copy(); xm2[i] -= 2*h
        grad[i] = (-F(xp2) + 8*F(xp1) - 8*F(xm1) + F(xm2)) / (12*h)
    return grad


def gradient_descent(F, gradF, x0, eps=1e-5, kmax=30000,
                     eta_fixed=None, beta=0.8):
    x = x0.copy().astype(float)
    k = 0

    while True:
        g = gradF(x)

        if eta_fixed is not None:
            eta = eta_fixed
        else:
            eta = 1.0
            p = 1
            grad_sq = np.dot(g, g)
            while (F(x - eta * g) > F(x) - (eta / 2.0) * grad_sq and p < 8):
                eta *= beta
                p += 1

        x = x - eta * g
        k += 1

        val = eta * np.linalg.norm(g)  # η‖∇F(x^k)‖ — fără calcul extra de gradient

        if val <= eps:
            return x, k, "convergent"

        if val > 1e10:
            return x, k, "divergent"

        if k >= kmax:
            return x, k, "kmax atins"


def make_numeric_grad(F):
    def ng(x):
        return numeric_grad(F, x)
    return ng


def run_all_strategies(F, gradF_analytic, x0, name, x_star_info,
                       eps=1e-5, kmax=30000):

    print(f"Functia: {name}")
    print(f"Solutia asteptata: {x_star_info}")
    print(f"Punct initial: x0 = {np.round(x0, 4)}")


    strategies = [
        ("Backtracking", None),
        ("eta = 1e-3",   1e-3),
        ("eta = 1e-4",   1e-4),
    ]

    gradF_numeric = make_numeric_grad(F)

    header = f"{'Strategia':<18} {'Gradient':<10} {'x_min':^32} {'Iteratii':>8}  {'Status'}"
    print(header)

    results = []
    for strat_name, eta_val in strategies:
        for grad_name, gradF in [("Analitic", gradF_analytic),
                                  ("Numeric",  gradF_numeric)]:
            x_min, iters, status = gradient_descent(
                F, gradF, x0, eps=eps, kmax=kmax, eta_fixed=eta_val
            )
            x_str = f"({x_min[0]:+.5f}, {x_min[1]:+.5f})"
            print(f"{strat_name:<18} {grad_name:<10} {x_str:^32} {iters:>8}  {status}")
            results.append((strat_name, grad_name, iters, status))

    bt_a = next(r for r in results if r[0] == "Backtracking" and r[1] == "Analitic")
    bt_n = next(r for r in results if r[0] == "Backtracking" and r[1] == "Numeric")
    print(f"Diferenta iteratii (Numeric - Analitic, backtracking): {bt_n[2]-bt_a[2]:+d}")
    return results



if __name__ == "__main__":
    np.random.seed(42)
    eps  = 1e-5
    kmax = 30000

    print(f"Precizie eps = {eps},  kmax = {kmax}")

    x0_F1 = np.random.uniform(-5, 5, 2)
    run_all_strategies(F1, grad_F1, x0_F1,
                       name="F1(x1,x2) = x1^2 + x2^2 - 2x1 - 4x2 - 1",
                       x_star_info="x* = (1, 2)")

    x0_F2 = np.random.uniform(-5, 5, 2)
    run_all_strategies(F2, grad_F2, x0_F2,
                       name="F2(x1,x2) = 3x1^2 - 12x1 + 2x2^2 + 16x2 - 10",
                       x_star_info="x* = (2, -4)")

    x0_F3_near = np.array([8.0, 4.0]) + np.random.uniform(-1, 1, 2)
    print(f"\nx0 langa x*: x0 = {np.round(x0_F3_near, 4)}")
    run_all_strategies(F3, grad_F3, x0_F3_near,
                       name="F3(x1,x2) = x1^2 - 4x1x2 + 4.5x2^2 - 4x2 + 3",
                       x_star_info="x* = (8, 4)")

    x0_F3_far = np.random.uniform(-5, 5, 2)
    print(f"\nx0 random (departe de x*): x0 = {np.round(x0_F3_far, 4)}")
    run_all_strategies(F3, grad_F3, x0_F3_far,
                       name="F3(x1,x2) = x1^2 - 4x1x2 + 4.5x2^2 - 4x2 + 3",
                       x_star_info="x* = (8, 4)  [posibila divergenta cu x0 departe]")

    x0_F4_near = np.array([-1.0, 0.5]) + np.random.uniform(-0.1, 0.1, 2)
    print(f"\nx0 langa x*: x0 = {np.round(x0_F4_near, 4)}")
    run_all_strategies(F4, grad_F4, x0_F4_near,
                       name="F4(x1,x2) = x1^2*x2 - 2x1*x2^2 + 3x1*x2 + 4",
                       x_star_info="x* = (-1, 0.5)  [minim local]")


    x0_l = np.random.uniform(-2, 2, 2)
    run_all_strategies(l_sigmoid, grad_l_sigmoid, x0_l,
                       name="l(w0,w1) = -ln(1-sigma(w0-w1)) - ln(sigma(w0+w1))",
                       x_star_info="nu exista minim finit")

    print(f"\n  {'Functia':<8} | {'Iter. Analitic':>14} | {'Iter. Numeric':>13} | {'Diferenta':>9}")

    np.random.seed(42)
    all_funcs = [
        (F1, grad_F1,               make_numeric_grad(F1), "F1", np.random.uniform(-5, 5, 2)),
        (F2, grad_F2,               make_numeric_grad(F2), "F2", np.random.uniform(-5, 5, 2)),
        (F3, grad_F3,               make_numeric_grad(F3), "F3", np.array([8.0, 4.0]) + np.random.uniform(-1, 1, 2)),
        (F4, grad_F4,               make_numeric_grad(F4), "F4", np.array([-1.0, 0.5]) + np.random.uniform(-0.1, 0.1, 2)),
        (l_sigmoid, grad_l_sigmoid, make_numeric_grad(l_sigmoid), "l",  np.random.uniform(-2, 2, 2)),
    ]

    for F, gradF_a, gradF_n, fname, x0 in all_funcs:
        _, ia, sa = gradient_descent(F, gradF_a, x0, eps=eps, kmax=kmax)
        _, in_, sn = gradient_descent(F, gradF_n, x0, eps=eps, kmax=kmax)
        note = "" if sa == "convergent" else f"({sa})"
        print(f"{fname:<8} | {ia:>14} | {in_:>13} | {in_-ia:>+9}  {note}")

Precizie eps = 1e-05,  kmax = 30000
Functia: F1(x1,x2) = x1^2 + x2^2 - 2x1 - 4x2 - 1
Solutia asteptata: x* = (1, 2)
Punct initial: x0 = [-1.2546  4.5071]
Strategia          Gradient                x_min               Iteratii  Status
Backtracking       Analitic         (+1.00000, +2.00000)              9  convergent
Backtracking       Numeric          (+1.00000, +2.00000)              9  convergent
eta = 1e-3         Analitic         (+0.99667, +2.00371)           3255  convergent
eta = 1e-3         Numeric          (+0.99667, +2.00371)           3255  convergent
eta = 1e-4         Analitic         (+0.96657, +2.03717)          21055  convergent
eta = 1e-4         Numeric          (+0.96657, +2.03717)          21055  convergent
Diferenta iteratii (Numeric - Analitic, backtracking): +0
Functia: F2(x1,x2) = 3x1^2 - 12x1 + 2x2^2 + 16x2 - 10
Solutia asteptata: x* = (2, -4)
Punct initial: x0 = [2.3199 0.9866]
Strategia          Gradient                x_min               Iteratii  Status
Ba